# CareLoop Clinics — No-Show Guard

## Part 1: Python Data Processing and API Enrichment

### Project Overview

CareLoop Clinics — No-Show Guard is a data analytics project designed to understand and address missed patient appointments.

This part focuses on loading, validating, cleaning and enriching appointment data using Python and external API information.

### Objectives
- Load and inspect clinic and appointment datasets.
- Validate records and identify data quality issues.
- Clean appointment data and separate rejected records.
- Enrich clinic information using the Open-Meteo API.
- Export cleaned and enriched datasets for further analysis.

### Tools and Libraries
- Python
- Pandas
- NumPy
- Requests
- Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1.1 — Data Loading and Initial Inspection

In this section, the original appointment and clinic datasets are loaded into Pandas DataFrames for further analysis.

In [ ]:
import csv

with open('/content/Appointments.csv', 'r') as file:
    reader = csv.DictReader(file)

    rows = list(reader)

print("Total records:", len(rows))
print("Column names:", reader.fieldnames)
print("First record:", rows[0])

Total records: 1020
Column names: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
First record: {'appointment_id': 'A0001', 'patient_id': 'P509', 'clinic_id': 'C05', 'booked_on': '2025-01-24', 'appt_date': '2025-01-25', 'age': '60', 'gender': 'F', 'distance_km': '18.0', 'fee_inr': '850', 'booking_channel': 'Phone', 'visit_type': 'Consultation', 'reminder_sent': 'Yes', 'prior_no_shows': '1', 'status': 'No-Show'}


In [ ]:
import pandas as pd

df = pd.read_csv('/content/Appointments.csv')

print("Dataset Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Information:")
df.info()

Dataset Shape: (1020, 14)

First 5 rows:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB


In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Missing values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

Total missing values:
37

Duplicate rows:
20

Data types:
appointment_id      object
patient_id          object
clinic_id           object
booked_on           object
appt_date           object
age                float64
gender              object
distance_km        float64
fee_inr              int64
booking_channel     object
visit_type          object
reminder_sent       object
prior_no_shows       int64
status              object
dtype: object


In [ ]:

import pandas as pd

df = pd.read_csv('/content/Appointments.csv')

print("Dataset Shape:", df.shape)
print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())


Dataset Shape: (1020, 14)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patie

## 1.2 — Data Cleaning and Validation

This section handles duplicate records, standardises booking channels, converts date columns, and addresses missing or invalid values.

In [34]:
import pandas as pd

# Load original dataset again
df = pd.read_csv('/content/Appointments.csv')

# Inspect dataset
print("Dataset Shape:", df.shape)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Shape: (1020, 14)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patie

In [35]:
# Remove exact duplicate rows
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1000


In [36]:
# Clean booking_channel
df['booking_channel'] = df['booking_channel'].str.strip().str.title()

print("Booking channel values after cleaning:")
print(df['booking_channel'].value_counts())

print("Rows after cleaning:", len(df))

Booking channel values after cleaning:
booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64
Rows after cleaning: 1000


In [37]:
# Convert date columns into datetime format

df['appt_date'] = pd.to_datetime(
    df['appt_date'], errors='coerce'
)

df['booked_on'] = pd.to_datetime(
    df['booked_on'], errors='coerce'
)

print("Rows after date conversion:", len(df))
print("\nMissing appointment dates:", df['appt_date'].isnull().sum())
print("Missing booking dates:", df['booked_on'].isnull().sum())

Rows after date conversion: 1000

Missing appointment dates: 6
Missing booking dates: 0


In [38]:
# Remove exact duplicate rows

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1000


In [39]:
# Remove rows with missing appointment dates or status

df = df.dropna(subset=['appt_date', 'status'])

print("Rows after removing missing values:", len(df))

Rows after removing missing values: 990


In [48]:
df['no_show'] = (df['status'] == 'no_show').astype(int)

### 1.2.1 — Handling Invalid and Missing Values

Invalid age values are treated as missing, and missing age and distance values are filled using their respective medians.

In [40]:
import numpy as np

# 1. Treat invalid ages as missing
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan

# 2. Fill missing age with median
df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)

# 3. Fill missing distance with median
df['distance_km'] = df['distance_km'].fillna(
    df['distance_km'].median()
)

# Check missing values
print("Missing age:", df['age'].isnull().sum())
print("Missing distance:", df['distance_km'].isnull().sum())

print("\nAge data type:", df['age'].dtype)

Missing age: 0
Missing distance: 0

Age data type: int64


In [41]:
# Calculate Q1, Q3 and IQR for distance_km

Q1_distance = df['distance_km'].quantile(0.25)
Q3_distance = df['distance_km'].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

print("Distance Q1:", Q1_distance)
print("Distance Q3:", Q3_distance)
print("Distance IQR:", IQR_distance)

Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375


In [ ]:
import numpy as np

### 1.2.2 — Distance Outlier Analysis

This section examines distance values to identify and assess potential outliers using statistical methods.

In [42]:
# Distance outlier treatment

Q1_distance = df['distance_km'].quantile(0.25)
Q3_distance = df['distance_km'].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

# Count outliers before capping
distance_capped = (
    (df['distance_km'] < lower_distance) |
    (df['distance_km'] > upper_distance)
).sum()

# Cap outliers
df['distance_km'] = np.clip(
    df['distance_km'],
    lower_distance,
    upper_distance
)

df['distance_km'] = df['distance_km'].round(1)


# Fee outlier treatment

Q1_fee = df['fee_inr'].quantile(0.25)
Q3_fee = df['fee_inr'].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

# Count outliers before capping
fee_capped = (
    (df['fee_inr'] < lower_fee) |
    (df['fee_inr'] > upper_fee)
).sum()

# Cap outliers
df['fee_inr'] = np.clip(
    df['fee_inr'],
    lower_fee,
    upper_fee
)

df['fee_inr'] = df['fee_inr'].round().astype(int)


print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Distance values capped: 26
Fee values capped: 3


In [43]:
# Replace invalid ages with missing values
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan

# Fill missing ages with median
df['age'] = df['age'].fillna(
    df['age'].median()
).round().astype(int)

print("Missing ages:", df['age'].isnull().sum())
print("Minimum age:", df['age'].min())
print("Maximum age:", df['age'].max())

Missing ages: 0
Minimum age: 18
Maximum age: 78


In [ ]:
# Create lead_days column
df['lead_days'] = (
    df['appt_date'] - df['booked_on']
).dt.days

# Create no_show column
df['no_show'] = (df['status'] == 'No-Show').astype(int)

# Check results
print(df[['booked_on', 'appt_date', 'lead_days', 'status', 'no_show']].head())

   booked_on  appt_date  lead_days    status  no_show
0 2025-01-24 2025-01-25          1   No-Show        1
1 2025-05-27 2025-06-06         10  Attended        0
2 2025-03-24 2025-03-30          6  Attended        0
4 2025-04-18 2025-04-22          4  Attended        0
5 2024-11-23 2025-01-02         40   No-Show        1


### Outlier Analysis

In [ ]:
# Select required columns
X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()

# Calculate mean of each column
means = np.mean(X, axis=0)

print("Column means:", means)

Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [ ]:
# Create Boolean mask
mask = df['lead_days'] > 14

# Count and share
print("Rows with lead_days > 14:", mask.sum())
print("Share of rows:", mask.mean())

Rows with lead_days > 14: 296
Share of rows: 0.298989898989899


In [ ]:
# Calculate z-scores
z_scores = (
    df['distance_km'] - np.mean(df['distance_km'])
) / np.std(df['distance_km'])

# Count rows where absolute z-score > 2
print("Rows with |z| > 2:", (np.abs(z_scores) > 2).sum())

Rows with |z| > 2: 53


## 1.3 — API Enrichment

In this section, clinic data is enriched with weather information using the Open-Meteo API.

In [7]:
import os

print(os.listdir('/content'))

['.config', 'clinics.csv', 'sample_data']


In [6]:
import shutil

shutil.move('/clinics.csv', '/content/clinics.csv')

'/content/clinics.csv'

In [8]:
import pandas as pd

clinics = pd.read_csv('/content/clinics.csv')

print(clinics.columns.tolist())
display(clinics.head())

['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873


In [10]:
import os

print(os.path.exists('/clinics.csv'))
print(os.path.exists('/content/clinics.csv'))

False
True


In [11]:
print(os.path.exists('/content/clinics_enriched.csv'))

False


In [12]:
print(clinics.columns.tolist())

['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


In [13]:
print('weather_df' in globals())
print('weather_data' in globals())

False
False


In [14]:
import requests

In [15]:
url = "https://api.open-meteo.com/v1/forecast"

In [16]:
weather_data = []

In [ ]:
for _, clinic in clinics.iterrows():
    print(clinic['clinic_name'])

Kothrud Family Clinic
Hadapsar Care Centre
Sitabuldi Health Point
Malviya Nagar Clinic
Vaishali Nagar Clinic
Edappally Medical Centre
Vijay Nagar Clinic
Gomti Nagar Care Point


In [ ]:
lat = clinic['latitude']
lon = clinic['longitude']

print(lat, lon)

26.8467 80.9462


In [ ]:
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": "temperature_2m",
    "forecast_days": 1
}

In [ ]:
response = requests.get(url, params=params, timeout=10)

print(response.status_code)

200


In [ ]:
data = response.json()

print(data.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])


In [ ]:
temperatures = data['hourly']['temperature_2m']

print(temperatures[:5])

[16.4, 15.8, 16.9, 19.0, 21.3]


In [ ]:
current_temp = temperatures[0]

print(current_temp)

16.4


In [ ]:
weather_data.append({
    "clinic_id": clinic['clinic_id'],
    "temperature": current_temp
})

print(weather_data)

[{'clinic_id': 'C08', 'temperature': 16.4}]


In [ ]:
print(len(weather_data))

1


In [ ]:
weather_data = []

In [ ]:
print("Total clinics processed:", len(weather_data))

Total clinics processed: 8


In [ ]:
for record in weather_data:
    print(record)

{'clinic_id': 'C01', 'temperature': 16.7}
{'clinic_id': 'C02', 'temperature': 18.0}
{'clinic_id': 'C03', 'temperature': 20.1}
{'clinic_id': 'C04', 'temperature': 17.4}
{'clinic_id': 'C05', 'temperature': 17.4}
{'clinic_id': 'C06', 'temperature': 24.9}
{'clinic_id': 'C07', 'temperature': 20.1}
{'clinic_id': 'C08', 'temperature': 16.4}


In [ ]:
weather_data = []

for _, clinic in clinics.iterrows():

    lat = clinic['latitude']
    lon = clinic['longitude']

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temperature = None

    for attempt in range(2):
        try:
            response = requests.get(
                url, params=params, timeout=10
            )

            if response.status_code == 200:
                data = response.json()
                temperatures = data.get(
                    'hourly', {}
                ).get('temperature_2m', [])

                if temperatures and temperatures[0] is not None:
                    temperature = temperatures[0]
                    break

        except requests.RequestException:
            pass

    weather_data.append({
        "clinic_id": clinic['clinic_id'],
        "temperature": temperature
    })

print(weather_data)

[{'clinic_id': 'C01', 'temperature': 16.7}, {'clinic_id': 'C02', 'temperature': 18.0}, {'clinic_id': 'C03', 'temperature': 20.1}, {'clinic_id': 'C04', 'temperature': 17.4}, {'clinic_id': 'C05', 'temperature': 17.4}, {'clinic_id': 'C06', 'temperature': 24.9}, {'clinic_id': 'C07', 'temperature': 20.1}, {'clinic_id': 'C08', 'temperature': 16.4}]


In [ ]:
print("Total clinics:", len(weather_data))

print("Missing temperatures:", sum(
    record['temperature'] is None
    for record in weather_data
))

Total clinics: 8
Missing temperatures: 0


In [ ]:
weather_df = pd.DataFrame(weather_data)

display(weather_df)

,clinic_id,temperature
0,C01,16.7
1,C02,18.0
2,C03,20.1
3,C04,17.4
4,C05,17.4
5,C06,24.9
6,C07,20.1
7,C08,16.4


In [21]:
clinics = clinics.merge(
    weather_df,
    on='clinic_id',
    how='left'
)

display(clinics.head())

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,18.3
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,20.0
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.4
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,16.2
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,16.2


In [ ]:
print(clinics.shape)
print(clinics.isnull().sum())

(8, 7)
clinic_id      0
clinic_name    0
city           0
tier           0
latitude       0
longitude      0
temperature    0
dtype: int64


In [17]:
weather_data = []

for _, clinic in clinics.iterrows():

    lat = clinic['latitude']
    lon = clinic['longitude']

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    current_temp = None

    for attempt in range(2):
        try:
            response = requests.get(
                url, params=params, timeout=10
            )

            if response.status_code == 200:
                data = response.json()
                temperatures = data['hourly']['temperature_2m']

                if temperatures:
                    current_temp = temperatures[0]
                break

        except (requests.RequestException, KeyError, IndexError):
            pass

    weather_data.append({
        "clinic_id": clinic['clinic_id'],
        "temperature": current_temp
    })

print(weather_data)

[{'clinic_id': 'C01', 'temperature': 18.3}, {'clinic_id': 'C02', 'temperature': 20.0}, {'clinic_id': 'C03', 'temperature': 20.4}, {'clinic_id': 'C04', 'temperature': 16.2}, {'clinic_id': 'C05', 'temperature': 16.2}, {'clinic_id': 'C06', 'temperature': 26.2}, {'clinic_id': 'C07', 'temperature': 16.6}, {'clinic_id': 'C08', 'temperature': 16.9}]


In [18]:
weather_df = pd.DataFrame(weather_data)
display(weather_df)

,clinic_id,temperature
0,C01,18.3
1,C02,20.0
2,C03,20.4
3,C04,16.2
4,C05,16.2
5,C06,26.2
6,C07,16.6
7,C08,16.9


In [ ]:
clinics = clinics.drop(columns=['temperature'], errors='ignore')

clinics = clinics.merge(
    weather_df,
    on='clinic_id',
    how='left'
)

display(clinics.head())

,clinic_id,clinic_name,city,tier,latitude,longitude,temperature
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,16.7
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.0
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.1
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,17.4
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,17.4


In [ ]:
print("Dataset shape:", clinics.shape)

print("\nMissing values:")
print(clinics.isnull().sum())

print("\nWeather data:")
display(clinics[['clinic_id', 'temperature']])

Dataset shape: (8, 7)

Missing values:
clinic_id      0
clinic_name    0
city           0
tier           0
latitude       0
longitude      0
temperature    0
dtype: int64

Weather data:


,clinic_id,temperature
0,C01,16.7
1,C02,18.0
2,C03,20.1
3,C04,17.4
4,C05,17.4
5,C06,24.9
6,C07,20.1
7,C08,16.4


In [19]:
import os
print(os.path.exists('/content/clinics_enriched.csv'))

False


In [20]:
weather_df = weather_df.rename(
    columns={'temperature': 'temp_c'}
)

print(weather_df.columns.tolist())

['clinic_id', 'temp_c']


In [ ]:
clinics = clinics.drop(
    columns=['temperature'],
    errors='ignore'
)

clinics = clinics.merge(
    weather_df,
    on='clinic_id',
    how='left'
)

print(clinics.columns.tolist())

['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']


In [22]:
print("Shape:", clinics.shape)
print("Columns:", clinics.columns.tolist())
print("\nMissing values:")
print(clinics.isnull().sum())

Shape: (8, 7)
Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']

Missing values:
clinic_id      0
clinic_name    0
city           0
tier           0
latitude       0
longitude      0
temp_c         0
dtype: int64


## 1.4 — Exporting and Verifying Output Files

In [23]:
clinics.to_csv('/content/clinics_enriched.csv', index=False)

In [24]:
print(os.path.exists('/content/clinics_enriched.csv'))

True


In [32]:
import os

for file in os.listdir('/content'):
    if file.endswith(('.csv', '.json', '.ipynb')):
        print(file)

rejected_rows.json
clinics_enriched.csv
clinics.csv
Appointments.csv


In [26]:
import os
print(os.path.exists('/content/rejected_rows.json'))

False


In [ ]:
print("rejected:", "rejected" in globals())
print("rejected_rows:", "rejected_rows" in globals())

rejected: True
rejected_rows: False


In [31]:
import csv
import json

accepted = []
rejected = []

def parse_row(row):
    row = row.copy()
    row['age'] = int(row['age'])
    row['fee_inr'] = int(row['fee_inr'])
    row['prior_no_shows'] = int(row['prior_no_shows'])
    row['distance_km'] = float(row['distance_km'])
    return row

with open('/content/Appointments.csv', 'r', newline='') as file:
    reader = csv.DictReader(file)

    print("Headers:", reader.fieldnames)

    for row in reader:
        try:
            converted_row = parse_row(row)
            accepted.append(converted_row)
        except ValueError:
            rejected.append(row)

with open('/content/rejected_rows.json', 'w') as file:
    json.dump(rejected, file, indent=2)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
print("JSON saved successfully!")




Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Accepted rows: 988
Rejected rows: 32
JSON saved successfully!


In [ ]:
import os
print(os.path.exists('/content/rejected_rows.json'))

True


In [ ]:
print("Appointments:", df.shape)
print("Clinics:", clinics.shape)

Appointments: (990, 16)
Clinics: (8, 7)


In [ ]:
import pandas as pd
import numpy as np
import datetime

clinic_cols = [
    'clinic_id', 'clinic_name', 'city',
    'tier', 'latitude', 'longitude', 'temp_c'
]

appt_cols = [
    'appointment_id', 'patient_id', 'clinic_id',
    'booked_on', 'appt_date', 'age', 'gender',
    'distance_km', 'fee_inr', 'booking_channel',
    'visit_type', 'reminder_sent', 'prior_no_shows',
    'status', 'lead_days', 'no_show'
]

def sql_value(value):
    if pd.isna(value):
        return "NULL"

    if isinstance(value, (pd.Timestamp, datetime.date, datetime.datetime)):
        return "'" + value.strftime("%Y-%m-%d") + "'"

    if isinstance(value, (np.integer, int)):
        return str(int(value))

    if isinstance(value, (np.floating, float)):
        return str(float(value))

    return "'" + str(value).replace("'", "''") + "'"


statements = []

for _, row in clinics[clinic_cols].iterrows():
    values = ", ".join(sql_value(row[col]) for col in clinic_cols)
    statements.append(
        f"INSERT INTO clinics ({', '.join(clinic_cols)}) VALUES ({values});"
    )

for _, row in df[appt_cols].iterrows():
    values = ", ".join(sql_value(row[col]) for col in appt_cols)
    statements.append(
        f"INSERT INTO appointments ({', '.join(appt_cols)}) VALUES ({values});"
    )

with open('/content/load_data.sql', 'w') as file:
    file.write('\n'.join(statements))

print("SQL file created successfully!")
print("Total INSERT statements:", len(statements))

SQL file created successfully!
Total INSERT statements: 998


In [ ]:
from google.colab import files
files.download('/content/load_data.sql')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###  1.5 - Clinic-wise Appointment Analysis

In [49]:
clinic_check = (
    df.groupby("clinic_id")
      .agg(
          appointments=("no_show", "count"),
          no_shows=("no_show", "sum")
      )
      .reset_index()
)

clinic_check["no_show_pct"] = (
    clinic_check["no_shows"] / clinic_check["appointments"] * 100
).round(1)

clinic_check

,clinic_id,appointments,no_shows,no_show_pct
0,C01,160,0,0.0
1,C02,143,0,0.0
2,C03,82,0,0.0
3,C04,142,0,0.0
4,C05,130,0,0.0
5,C06,142,0,0.0
6,C07,101,0,0.0
7,C08,90,0,0.0


In [50]:
clinic_check[["appointments", "no_shows"]].sum()

,0
appointments,990
no_shows,0


In [51]:
print(df['status'].unique())

['No-Show' 'Attended']


In [52]:
df['no_show'] = (df['status'] == 'No-Show').astype(int)

In [53]:
print(df['no_show'].sum())

218
